## ESTA ES OFICIAL DE LAS CLASES

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, learning_curve, validation_curve
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    "train": "#1f77b4",
    "val": "#ff7f0e",
    "precision": "#2ca02c",
    "recall": "#d62728",
    "f1": "#9467bd",
    "roc": "#17becf"
}

def evaluate_model(name, pipeline, param_grid=None, complexity_param=None, complexity_range=None):

    print(f"\n{'='*60}")
    print(f"Modelo: {name}")
    print(f"{'='*60}")

    # GridSearch
    if param_grid:
        grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, scoring='accuracy')
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_
        print(f"Mejores parámetros: {grid_search.best_params_}")
    else:
        model = pipeline
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, y_pred)

    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle(f"{name} - Evaluación completa", fontsize=18, fontweight='bold')

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    ax = axes[0, 0]
    im = ax.imshow(cm, cmap='Blues')

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j],
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black",
                    fontsize=12)

    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    ax = axes[0, 1]
    ax.plot(fpr, tpr, color=COLORS["roc"], lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
    ax.set_title("ROC Curve")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()

    # Precision-Recall
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

    ax = axes[0, 2]
    ax.plot(recall, precision, color=COLORS["precision"], lw=2)
    ax.set_title("Precision-Recall Curve")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")

    # Threshold curves
    f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
    best_idx = f1_scores.argmax()
    best_thresh = thresholds[best_idx]

    ax = axes[1, 0]
    ax.plot(thresholds, precision[:-1], label="Precision", color=COLORS["precision"])
    ax.plot(thresholds, recall[:-1], label="Recall", color=COLORS["recall"])
    ax.plot(thresholds, f1_scores, label="F1", color=COLORS["f1"])
    ax.axvline(best_thresh, color='gray', linestyle='--', alpha=0.7)
    ax.set_title(f"Metrics vs Threshold (best={best_thresh:.2f})")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.legend()

    # Learning Curve
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5)
    )

    ax = axes[1, 1]
    ax.plot(train_sizes, train_scores.mean(axis=1), marker='o',
            color=COLORS["train"], label="Train")
    ax.plot(train_sizes, val_scores.mean(axis=1), marker='o',
            color=COLORS["val"], label="Validation")

    ax.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1),
                    alpha=0.1, color=COLORS["train"])

    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1),
                    alpha=0.1, color=COLORS["val"])

    ax.set_title("Learning Curve")
    ax.set_xlabel("Training Size")
    ax.set_ylabel("Accuracy")
    ax.legend()

    # Complexity Curve
    ax = axes[1, 2]

    if complexity_param and complexity_range:
        train_scores, val_scores = validation_curve(
            model, X, y,
            param_name=complexity_param,
            param_range=complexity_range,
            cv=5,
            n_jobs=-1
        )

        ax.plot(complexity_range, train_scores.mean(axis=1),
                marker='o', color=COLORS["train"], label="Train")

        ax.plot(complexity_range, val_scores.mean(axis=1),
                marker='o', color=COLORS["val"], label="Validation")

        ax.set_xscale("log" if max(complexity_range) / min(complexity_range) > 10 else "linear")
        ax.set_title("Complexity Curve")
        ax.set_xlabel(complexity_param.split("__")[-1])
        ax.set_ylabel("Accuracy")
        ax.legend()
    else:
        ax.set_title("No complexity parameter")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


### FUNCIONES QUE PUEDEN SERVIR PARA EL TP

Función 1: seleccionar un modelo

In [ ]:
from sklearn.model_selection import GridSearchCV

def seleccionar_modelo(nombre, pipeline, param_grid, X_train, y_train):
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print(f"Modelo: {nombre}")
    print("Mejores hiperparámetros:", grid.best_params_)
    print("Mejor Macro F1 promedio:", grid.best_score_)

    return grid

Función 2: evaluar el modelo seleccionado

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

def evaluar_modelo(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)

    print(f"Modelo: {nombre}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()